In [1]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import root_scalar

def calculate_garch_tail_index(alpha: float, beta: float) -> float:
    """
    Calculates the theoretical tail index (kappa) for a GARCH(1,1) process
    under standard Gaussian innovations.
    """
    def integrand(z, a, b, k):
        # The core expectation: ((alpha * z^2 + beta)^(kappa/2)) * phi(z)
        return ((a * z**2 + b)**(k / 2.0)) * np.exp(-z**2 / 2.0) / np.sqrt(2 * np.pi)

    def objective(k, a, b):
        # Exploit the symmetry of the normal distribution: integrate 0 to +inf and multiply by 2
        val, _ = quad(integrand, 0, np.inf, args=(a, b, k))
        return val * 2.0 - 1.0

    # Ensure covariance stationarity before searching
    if alpha + beta >= 1.0:
        raise ValueError("Process is not strictly covariance stationary (alpha + beta >= 1).")

    # Brent's method to find the root. Bracket assumes kappa > 2 for stationary processes.
    try:
        res = root_scalar(objective, args=(alpha, beta), bracket=[2.01, 100], method='brentq')
        return res.root
    except ValueError as e:
        print(f"Failed to find root in bracket for alpha={alpha}, beta={beta}: {e}")
        return np.nan

pairs = [(0.07, 0.63), (0.08, 0.72), (0.09, 0.81), (0.092, 0.828), (0.095, 0.855), (0.098, 0.882)]
for a, b in pairs:
    kappa = calculate_garch_tail_index(a, b)
    print(f"Alpha: {a:.3f}, Beta: {b:.3f} -> Tail Index: {kappa:.2f}")

Alpha: 0.070, Beta: 0.630 -> Tail Index: 26.46
Alpha: 0.080, Beta: 0.720 -> Tail Index: 20.44
Alpha: 0.090, Beta: 0.810 -> Tail Index: 14.08
Alpha: 0.092, Beta: 0.828 -> Tail Index: 12.49
Alpha: 0.095, Beta: 0.855 -> Tail Index: 9.64
Alpha: 0.098, Beta: 0.882 -> Tail Index: 5.74
